# Day 5 — ILT 1: Transformation Patterns — Standardize, Conform, Enrich

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Reads from** | `gbmart.bronze.*` (all 4 sources) |
| **Feeds into** | Day 5 hands-on labs — Silver layer build |
| **Duration** | 60 minutes |
| **Catalog** | `gbmart` (Unity Catalog) |

### Learning Objectives
- Tell apart three transformation categories: **standardize**, **conform**, **enrich**
- Recognize *why* each one exists as a separate step, not just "cleaning"
- See real examples from GlobalMart's own Bronze data — not toy examples
- Understand the rule: *derive a business value once, in Silver, reuse everywhere*

---
This is a lecture notebook — every code cell is a real, already-solved example pulled from GlobalMart's actual Bronze tables. You'll build the full Silver layer yourself in the Day 5 hands-on labs that follow.

## Why Bronze Isn't Enough On Its Own

Bronze does exactly one job well: it lands raw source data with zero data loss, in whatever shape the source gave it to us. That's the right job for Bronze — but it means Bronze data is usually **not safe to hand to a business user or a report** as-is.

GlobalMart's Bronze layer has four sources, ingested two different ways:

| Source | Ingestion path | Column naming as it lands in Bronze |
|---|---|---|
| `orders`, `order_items` (Postgres/Supabase) | **Lakeflow Connect** (CDC) | Lakeflow **lowercases everything** — `OrderID` → `orderid` |
| `customers`, `products`, `address`, `payments`, `payment_methods` (ADLS files) | **Autoloader** | Keeps the **source file's original casing** — `CustomerID` stays `CustomerID` |

Two ingestion tools, two different naming conventions, landing in the *same* catalog. If Gold tries to join `bronze.orders.customerid` to `bronze.customers.CustomerID` directly, every query becomes a guessing game about which table uses which casing. That's one of the three problems this notebook solves — **conform**. The other two are **standardize** (fix values, not just names) and **enrich** (add business value that doesn't exist yet).

---
## 1. Standardize — Fix the *Values*, Not Just the Shape

Standardizing means the data is *technically* usable (right table, right column) but the actual values are inconsistent, malformed, or in the wrong format for downstream use — wrong data type, extra whitespace, inconsistent casing, encoded values that need decoding.

### Real example — `PhoneNumber` in `bronze.customers`

Every phone number in Bronze is 12 digits: GlobalMart's registration form auto-prepends the `+91` India country code but stores it **without the `+`** — so `+91-7584314890` lands in Bronze as `917584314890`. It's not wrong data, it's an unstandardized *format*.

In [ ]:
from pyspark.sql.functions import col, length, concat, lit, when

bronze_customers = spark.table("gbmart.bronze.customers")

# PhoneNumber is stored as BIGINT in Bronze (Autoloader inferred it as a number) —
# cast to string FIRST, or the leading digits get treated as arithmetic, not text.
standardized_phone = (
    bronze_customers
    .withColumn("PhoneNumber", col("PhoneNumber").cast("string"))
    .withColumn(
        "PhoneNumber",
        # Only reformat numbers that actually match the 12-digit '91' pattern —
        # never blindly slice a string without confirming its shape first.
        when(
            (length(col("PhoneNumber")) == 12) & col("PhoneNumber").startswith("91"),
            concat(lit("+91-"), col("PhoneNumber").substr(3, 10))
        ).otherwise(col("PhoneNumber"))
    )
)

standardized_phone.select("CustomerID", "PhoneNumber").show(5, truncate=False)

### Real example — Email formatting errors

Two distinct, recoverable patterns show up in `bronze.customers.Email`:

| Pattern | Example | Root cause |
|---|---|---|
| Accidental space | `swaminathaninaaya 482@outlook.com` | Mobile keyboard autocorrect inserts a space after a long name |
| Apostrophe in surname | `taran.d'alia307@hotmail.com` | Surnames like D'Souza/D'Silva include `d'` — most email providers strip it, this form didn't |

**Standardize, don't discard.** Both are fixable formatting problems, not bad data — quarantining these customers would mean losing real people over a keyboard quirk. That distinction (fixable format vs. genuinely bad value) is exactly what separates *standardize* from a *data quality* decision — more on that split in ILT 2.

In [ ]:
from pyspark.sql.functions import regexp_replace, trim, lower

standardized_email = bronze_customers.withColumn(
    "Email",
    regexp_replace(
        regexp_replace(
            trim(lower(col("Email"))),
            " ", ""                        # Pattern 1: remove the stray space
        ),
        "d['\u2019\u2018]", ""              # Pattern 2: remove d + any apostrophe variant
    )
)

standardized_email.select("CustomerID", "Email").show(5, truncate=False)

---
## 2. Conform — One Shared Vocabulary Across All 4 Sources

Conforming means making data from *different sources* speak the same structural language — same column naming convention, same casing, same units — so that Gold can join across them without translation logic scattered everywhere.

GlobalMart's rule: **every Silver table uses lowercase `snake_case` column names**, regardless of how the source or ingestion tool spelled them.

In [ ]:
# Lakeflow Connect (CDC) already lowercases Postgres column names —
# 'orderid' just needs the underscore added to become snake_case.
bronze_orders = spark.table("gbmart.bronze.orders")
conformed_orders = (
    bronze_orders
    .withColumnRenamed("orderid",      "order_id")
    .withColumnRenamed("customerid",   "customer_id")
    .withColumnRenamed("orderdate",    "order_date")
    .withColumnRenamed("orderchannel", "order_channel")
)

# Autoloader kept the source file's original PascalCase — a bigger rename.
conformed_customers = (
    bronze_customers
    .withColumnRenamed("CustomerID",  "customer_id")
    .withColumnRenamed("FirstName",   "first_name")
    .withColumnRenamed("LastName",    "last_name")
    .withColumnRenamed("Email",       "email")
    .withColumnRenamed("PhoneNumber", "phone_number")
)

print("orders   ->", [c for c in conformed_orders.columns if c in ("order_id","customer_id","order_date","order_channel")])
print("customers->", [c for c in conformed_customers.columns if c in ("customer_id","first_name","last_name","email","phone_number")])

### Real example — conforming a *value*, not just a column name

`bronze.address` has the same underscore-for-space corruption in both `State` and `AddressType` (`Tamil_Nadu` instead of `Tamil Nadu`). This is a conform step too — it's not fixing a *wrong* value, it's making the representation of a known value consistent with how every other Silver table writes its text fields.

In [ ]:
bronze_address = spark.table("gbmart.bronze.addresses")

conformed_address = bronze_address \
    .withColumn("State", regexp_replace(col("State"), "_", " ")) \
    .withColumn("AddressType", regexp_replace(col("AddressType"), "_", " "))

conformed_address.select("AddressID", "State", "AddressType").distinct().show(5, truncate=False)

---
## 3. Enrich — Add Business Value That Doesn't Exist Yet

Enriching means deriving new columns that answer a business question the raw data can't answer on its own — a computed metric, a category, a flag.

### The rule: compute once in Silver, reuse everywhere in Gold

Take `customer_tenure_days` — "how many days has this customer been with GlobalMart." Gold reports, BI dashboards, and Genie will all eventually ask for customer segments like *New (< 90 days)*, *Regular (90–365 days)*, *Loyal (> 365 days)*. If tenure isn't computed in Silver, **every** downstream consumer repeats `datediff(current_date(), registration_date)` themselves — the same logic duplicated in five different places, with five chances to get it subtly wrong.

**If multiple Gold tables/reports need the same derived value, compute it once, in Silver.**

In [ ]:
from pyspark.sql.functions import floor, datediff, current_date, to_date

enriched_customers = (
    bronze_customers
    .withColumn("DateOfBirth",      to_date(col("DateOfBirth"),      "yyyy-MM-dd"))
    .withColumn("RegistrationDate", to_date(col("RegistrationDate"), "yyyy-MM-dd"))
    # age and tenure are both derived once here — Gold never recomputes them
    .withColumn("age",                  floor(datediff(current_date(), col("DateOfBirth")) / 365.25).cast("int"))
    .withColumn("customer_tenure_days", datediff(current_date(), col("RegistrationDate")).cast("int"))
)

enriched_customers.select("CustomerID", "age", "customer_tenure_days").show(5)

### Real example — order fulfillment KPIs

`bronze.orders` has four raw date columns (`orderdate`, `shippingdate`, `expecteddeliverydate`, `actualdeliverydate`). None of them is itself a business metric — but the *gaps between them* are exactly what Operations and the delivery-SLA dashboard need every day.

In [ ]:
from pyspark.sql.types import DateType

enriched_orders = (
    bronze_orders
    .withColumn("orderdate",            col("orderdate").cast(DateType()))
    .withColumn("shippingdate",         col("shippingdate").cast(DateType()))
    .withColumn("expecteddeliverydate", col("expecteddeliverydate").cast(DateType()))
    .withColumn("actualdeliverydate",   col("actualdeliverydate").cast(DateType()))

    # Fulfillment SLA KPI: how many days between order and shipment
    .withColumn("order_to_ship_days",
        when(col("shippingdate").isNotNull(), datediff(col("shippingdate"), col("orderdate")))
        .otherwise(lit(None).cast("int"))
    )
    # Customer-experience KPI: was delivery later than promised?
    .withColumn("is_late",
        when(
            col("actualdeliverydate").isNotNull() & col("expecteddeliverydate").isNotNull(),
            col("actualdeliverydate") > col("expecteddeliverydate")
        ).otherwise(lit(None).cast("boolean"))
    )
    # Single readable status — Genie and BI tools shouldn't have to infer this from raw dates
    .withColumn("order_status",
        when(col("actualdeliverydate").isNotNull(), lit("Delivered"))
        .when(col("shippingdate").isNotNull(),      lit("Shipped"))
        .otherwise(                                  lit("Pending"))
    )
)

enriched_orders.select("orderid", "order_to_ship_days", "is_late", "order_status").show(5)

---
## Putting It Together: The Order Matters

In practice these three steps aren't strictly sequential — but a useful default order is:

1. **Standardize** first — fix values while they're still easy to reason about (before renames make the logic harder to read)
2. **Conform** next — once values are trustworthy, align naming/casing so joins across sources are simple
3. **Enrich** last — derive new columns on top of data that's already standardized and conformed, so the derived logic doesn't have to work around messy inputs

One thing conspicuously **not** covered yet: what happens to rows that *can't* be fixed — a null primary key, an impossible date, an out-of-range rating? That's Data Quality Constraints & Validation — ILT 2, next.